<a href="https://colab.research.google.com/github/marcoastupp/feb-assets-pipeline/blob/main/FEB_Upscaler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 1. INSTALAÇÃO E PATCH DE COMPATIBILIDADE (COLAB T4)
# ==============================================================================
print("⚙️ Instalando dependências do Real-ESRGAN...")
!pip install -q basicsr facexlib gfpgan realesrgan

# Patch automático para compatibilidade com torchvision recente no Colab
import glob
import os

for file_path in glob.glob('/usr/local/lib/python*/dist-packages/basicsr/data/degradations.py'):
    with open(file_path, 'r') as f:
        code = f.read()
    if 'functional_tensor' in code:
        code = code.replace(
            'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
            'from torchvision.transforms.functional import rgb_to_grayscale'
        )
        with open(file_path, 'w') as f:
            f.write(code)
        print("✅ Patch de compatibilidade aplicado no basicsr!")

# ==============================================================================
# 2. MONTAGEM DO GOOGLE DRIVE E CONFIGURAÇÃO DE PASTAS
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

# Defina as pastas no seu Google Drive (crie-as se ainda não existirem)
PASTA_ENTRADA = '/content/drive/MyDrive/FEB_Assets/01_Base_Flux'
PASTA_SAIDA   = '/content/drive/MyDrive/FEB_Assets/02_Upscaled_4x'

os.makedirs(PASTA_ENTRADA, exist_ok=True)
os.makedirs(PASTA_SAIDA, exist_ok=True)

print(f"📂 Lendo imagens de: {PASTA_ENTRADA}")
print(f"💾 Salvando resultados em: {PASTA_SAIDA}")

# ==============================================================================
# 3. EXECUÇÃO DO UPSCALE (4X COM REAL-ESRGAN)
# ==============================================================================
import urllib.request

# Baixa os pesos oficiais do modelo fotorrealista (RealESRGAN_x4plus)
weights_dir = '/content/weights'
os.makedirs(weights_dir, exist_ok=True)
model_path = os.path.join(weights_dir, 'RealESRGAN_x4plus.pth')

if not os.path.exists(model_path):
    print("⬇️ Baixando pesos do RealESRGAN_x4plus...")
    urllib.request.urlretrieve(
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
        model_path
    )

# Roda o script oficial de inferência em lote (tile 512 evita estourar VRAM)
# Obs: Use --outscale 2 se quiser dobrar a resolução (ex: 1024 -> 2048)
# ou --outscale 4 para quadruplicar (ex: 1024 -> 4096).
FATOR_ESCALA = 2  # Recomendado: 2 (gera imagens ~2K ultra nítidas e rápidas)

print(f"🚀 Iniciando Upscale ({FATOR_ESCALA}x)...")
!python /usr/local/lib/python*/dist-packages/realesrgan/inference_realesrgan.py \
    -n RealESRGAN_x4plus \
    -i "{PASTA_ENTRADA}" \
    -o "{PASTA_SAIDA}" \
    --model_path "{model_path}" \
    --outscale {FATOR_ESCALA} \
    --tile 512 \
    --fp32

print("✨ Processo concluído! Confira os arquivos em alta resolução no seu Google Drive.")